
# LA Crime Data — Exploratory Data Analysis & Visualization

This notebook performs exploratory data analysis (EDA) on the **cleaned LAPD crime dataset covering crime occurrence from January 1, 2010 through December 31, 2024**.

The cleaned analytical dataset is loaded from the Parquet output produced by the data-cleaning notebook. This notebook is intentionally focused on **understanding patterns, distributions, trends, relationships, and unusual observations** rather than modifying the source data.

## Visualization approach

The notebook uses **Plotly Express** for interactive visualizations. Wherever possible, the data is aggregated before plotting so that charts remain readable and performant despite the size of the dataset.

## Main EDA questions

1. How does reported crime volume change over time?
2. Which crime types, areas, premises, and weapons are most common?
3. How do crime patterns vary by month, weekday, and time of day?
4. What does the victim-age, sex, and descent distribution look like?
5. How does reporting lag vary over time and across crime categories?
6. What geographic patterns can be observed from LAPD area and coordinate data?
7. Which relationships or unusual patterns deserve deeper investigation in later analysis?

> **Important:** This is an exploratory notebook. Observed associations are described as patterns in the dataset and are not treated as causal relationships.


In [ ]:
# Import the libraries used throughout the EDA notebook.
# Pandas handles tabular data and aggregation, while Plotly Express produces
# interactive charts directly from pandas DataFrames.

from pathlib import Path

import pandas as pd
import plotly.express as px

In [ ]:
# Configure pandas display options for readable inspection of DataFrames.
# These settings affect display only; they do not modify the analytical data.

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

## 1. Load the Cleaned Analytical Dataset

The previous notebook completed the source-data cleaning, validation, feature engineering, and Parquet export. The EDA notebook therefore starts from that **validated analytical output** instead of re-cleaning the raw CSV files.

Using the processed Parquet file keeps this notebook focused on analysis and ensures that the EDA uses the same dataset that passed the cleaning notebook's validation gates.

In [ ]:
# Define the processed-data path used by the project structure.
# If this notebook is moved to a different directory, update only this path cell.

DATA_PATH = Path("../data/processed/crime_2010_2024.parquet")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found at: {DATA_PATH.resolve()}\n"
        "Check that the cleaning notebook has produced the Parquet file and "
        "that this notebook is running from the expected project directory."
    )

# Load the validated analytical dataset.
df = pd.read_parquet(DATA_PATH)

print(f"Dataset loaded successfully: {DATA_PATH.resolve()}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

In [ ]:
# Inspect the analytical schema before beginning the EDA.
# This checkpoint confirms that the expected engineered temporal features are present.

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

In [ ]:
# Define the canonical schema independently of the loaded DataFrame.
# These are the 29 columns produced by the cleaning pipeline.

EXPECTED_COLUMNS = [
    "DR_NO",
    "Date Rptd",
    "DATE OCC",
    "TIME OCC",
    "AREA",
    "AREA NAME",
    "Rpt Dist No",
    "Crm Cd",
    "Crm Cd Desc",
    "Vict Age",
    "Vict Sex",
    "Vict Descent",
    "Premis Cd",
    "Premis Desc",
    "Weapon Used Cd",
    "Weapon Desc",
    "Status",
    "Status Desc",
    "LOCATION",
    "LAT",
    "LON",
    "year",
    "month",
    "month_name",
    "day_of_week",
    "day_name",
    "hour",
    "minute",
    "reporting_lag_days",
]

# Compare the expected schema with the actual dataset schema.
expected_columns = set(EXPECTED_COLUMNS)
actual_columns = set(df.columns)

missing_columns = sorted(expected_columns - actual_columns)
unexpected_columns = sorted(actual_columns - expected_columns)

# Fail immediately if the dataset schema has changed.
# Report both missing and unexpected columns to help diagnose the issue.
if missing_columns or unexpected_columns:
    error_messages = [
        "EDA input schema validation failed."
    ]

    if missing_columns:
        error_messages.append(
            f"Missing required columns: {missing_columns}"
        )

    if unexpected_columns:
        error_messages.append(
            f"Unexpected columns: {unexpected_columns}"
        )

    raise ValueError("\n".join(error_messages))

# Validate important data integrity assumptions.
assert df["DR_NO"].is_unique, (
    "Duplicate incident numbers found in DR_NO."
)

assert df["DATE OCC"].is_monotonic_increasing, (
    "DATE OCC is not sorted in ascending order."
)

# Report successful validation.
print("EDA input schema validation passed.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")



## 2. Analytical Data Quality Overview

The cleaning notebook already handled invalid values and documented remaining missingness. Here, missingness is treated as an **EDA characteristic** rather than something to automatically fix.

This section helps identify fields whose coverage may affect later visualizations. For example, weapon and coordinate fields contain meaningful source missingness, so percentages involving those fields should be interpreted using the available observations rather than assuming that missing means zero.

In [ ]:
# Summarize missing values in the dataset, from highest to lowest percentage.

missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing (%)": df.isna().mean().mul(100).round(2),
    "Available Count": df.notna().sum()
}).sort_values("Missing (%)", ascending=False)

display(missing_summary)

In [ ]:
# Summarize unique values and missing data in key categorical columns.

categorical_columns = [
    "AREA NAME", "Crm Cd Desc", "Vict Sex", "Vict Descent",
    "Premis Desc", "Weapon Desc", "Status Desc"
]

category_summary = pd.DataFrame({
    "Unique Values": df[categorical_columns].nunique(),
    "Missing (%)": (df[categorical_columns].isna().mean() * 100).round(2)
}).sort_values("Unique Values", ascending=False)

display(category_summary)

## 3. Crime Volume Over Time

Temporal analysis is the first major EDA layer because the dataset spans fifteen years. We will examine the number of recorded incidents by **year, month, and day of week**.

These charts describe recorded incident volume in the cleaned dataset. Changes can reflect many factors, including actual crime patterns, reporting practices, population/activity changes, administrative processes, or changes in the source system.

In [ ]:
# Count incidents per year.
yearly_counts = df.groupby("year").size().reset_index(name="incident_count")

# Create an interactive line chart.
px.line(
    yearly_counts,
    x="year",
    y="incident_count",
    markers=True,
    title="Recorded Crime Incidents by Year (2010-2024)",
    labels={"year": "Year", "incident_count": "Number of Incidents"}
)

In [ ]:
# Count incidents by month and keep months in calendar order.
monthly_counts = (
    df.groupby(["month", "month_name"])
    .size()
    .reset_index(name="incident_count")
    .sort_values("month")
)

# Create and display the bar chart directly.
px.bar(
    monthly_counts,
    x="month_name",
    y="incident_count",
    title="Recorded Crime Incidents by Calendar Month",
    labels={"month_name": "Month", "incident_count": "Number of Incidents"},
    text_auto=True,
    category_orders={"month_name": monthly_counts["month_name"].tolist()}
)

In [ ]:
# Count incidents by day of week, ordered Monday through Sunday.
weekday_counts = (
    df.groupby(["day_of_week", "day_name"])
    .size()
    .reset_index(name="incident_count")
    .sort_values("day_of_week")
)

# Create and display the bar chart.
px.bar(
    weekday_counts,
    x="day_name",
    y="incident_count",
    title="Recorded Crime Incidents by Day of Week",
    labels={"day_name": "Day of Week", "incident_count": "Number of Incidents"},
    text_auto=True,
    category_orders={"day_name": weekday_counts["day_name"].tolist()}
)

In [ ]:
# Create a year-by-month matrix to analyze seasonal crime patterns over time.
# Aggregate incident counts by year and month, then reshape the data for the heatmap.

year_month = (
    df.groupby(["year", "month"], as_index=False)
      .size()
      .rename(columns={"size": "incident_count"})
)

# Convert the grouped data into a matrix and fill missing combinations with zero.
year_month_pivot = year_month.pivot(
    index="year", columns="month", values="incident_count"
).fillna(0)

# Reshape the matrix into long format for Plotly visualization.
year_month_long = (
    year_month_pivot.reset_index()
    .melt(id_vars="year", var_name="month", value_name="incident_count")
)

# Create a heatmap to visualize crime volume across years and months.
px.density_heatmap(
    year_month_long,
    x="month",
    y="year",
    z="incident_count",
    histfunc="sum",
    title="Crime Volume by Year and Month",
    labels={
        "month": "Calendar Month",
        "year": "Occurrence Year",
        "incident_count": "Incidents"
    },
    text_auto=True
).update_xaxes(dtick=1)


## 4. Crime-Type Distribution

The primary crime code and description represent the main incident classification retained in the cleaned dataset.

The first view focuses on the most frequent crime descriptions. Limiting the chart to the top categories keeps the comparison readable while the underlying table can still be used to inspect the complete distribution.


In [ ]:
# Count incidents for each primary crime description and identify the most frequent crime types.
# Limit the results to the top N categories to keep the analysis readable.

TOP_N = 10

top_crimes = (
    df["Crm Cd Desc"]
    .value_counts(dropna=False)
    .rename_axis("Crm Cd Desc")
    .reset_index(name="incident_count")
    .head(TOP_N)
)

# Display the top crime types and their incident counts.
display(top_crimes)

# Create a horizontal bar chart to compare incident counts across the top crime types.
px.bar(
    top_crimes.sort_values("incident_count"),
    x="incident_count",
    y="Crm Cd Desc",
    orientation="h",
    text_auto=True,
    title=f"Top {TOP_N} Primary Crime Types by Recorded Incidents",
    labels={
        "Crm Cd Desc": "Crime Type",
        "incident_count": "Number of Incidents"
    },
)

In [ ]:
# Select the five most common crime types to analyze their annual trends.
top5_crimes = top_crimes.head(5)["Crm Cd Desc"].tolist()

# Count incidents for each selected crime type by year.
crime_year = (
    df[df["Crm Cd Desc"].isin(top5_crimes)]
    .groupby(["year", "Crm Cd Desc"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
)

# Create a line chart to compare yearly incident trends across the five crime types.
px.line(
    crime_year,
    x="year",
    y="incident_count",
    color="Crm Cd Desc",
    markers=True,
    title="Annual Trends for the Five Most Common Crime Types",
    labels={
        "year": "Occurrence Year",
        "incident_count": "Number of Incidents",
        "Crm Cd Desc": "Crime Type"
    },
).update_layout(hovermode="x unified")


## 5. LAPD Area Patterns

The dataset contains both the numeric `AREA` identifier and the human-readable `AREA NAME`. For EDA, `AREA NAME` is easier to interpret.

This section compares overall incident volume across areas and then examines how area-level volume changes over time.


In [ ]:
# Rank LAPD areas by the total number of recorded crime incidents.
area_counts = (
    df["AREA NAME"]
    .value_counts(dropna=False)
    .rename_axis("AREA NAME")
    .reset_index(name="Incident Count")
    .sort_values("Incident Count", ascending=False)
)

# Display the incident counts for each LAPD area.
display(area_counts)

# Create a horizontal bar chart to compare recorded incidents across LAPD areas.
px.bar(
    area_counts.sort_values("Incident Count"),
    x="Incident Count",
    y="AREA NAME",
    orientation="h",
    title="Recorded Crime Incidents by LAPD Area",
    labels={
        "AREA NAME": "LAPD Area",
        "Incident Count": "Number of Incidents"
    },
)

In [ ]:
# Summarize recorded crime incidents by year and LAPD area.
# This allows comparisons of crime volume across areas and over time.

area_year = (
    df.groupby(["year", "AREA NAME"], as_index=False)
      .size()
      .rename(columns={"size": "Incident Count"})
)

# Create a heatmap to visualize annual crime counts for each LAPD area.
px.density_heatmap(
    area_year,
    x="year",
    y="AREA NAME",
    z="Incident Count",
    histfunc="sum",
    title="Crime Volume by Year and LAPD Area",
    labels={
        "year": "Occurrence Year",
        "AREA NAME": "LAPD Area",
        "Incident Count": "Incidents"
    },
).update_xaxes(dtick=1)


## 6. Time-of-Day Patterns

The cleaned dataset derives `hour` and `minute` from the source `TIME OCC` field. These features allow us to examine when incidents most frequently occurred.

Because `TIME OCC` is recorded at incident level, the first visualizations aggregate incidents by hour rather than plotting millions of individual observations.


In [ ]:
# Count recorded crime incidents for each hour of the day across the study period.
hour_counts = (
    df.groupby("hour", as_index=False)
      .size()
      .rename(columns={"size": "incident_count"})
      .sort_values("hour")
)

# Create a line chart to show how incident counts vary throughout the day.
px.line(
    hour_counts,
    x="hour",
    y="incident_count",
    markers=True,
    title="Recorded Crime Incidents by Hour of Day",
    labels={"hour": "Hour of Day", "incident_count": "Number of Incidents"
    }).update_xaxes(dtick=1)

In [ ]:
# Group occurrence hours into four broad time-of-day periods.
# These categories are created for analysis and are not original LAPD fields.

def time_of_day(hour):
    if pd.isna(hour):
        return pd.NA
    hour = int(hour)
    if 0 <= hour < 6:
        return "Night (00-05)"
    if 6 <= hour < 12:
        return "Morning (06-11)"
    if 12 <= hour < 18:
        return "Afternoon (12-17)"
    return "Evening (18-23)"

# Assign each incident to its corresponding time-of-day group.
df["time_of_day"] = df["hour"].map(time_of_day).astype("string")

# Count incidents in each time-of-day group, including missing values.
time_of_day_counts = (
    df["time_of_day"]
    .value_counts(dropna=False)
    .rename_axis("time_of_day")
    .reset_index(name="incident_count")
)

# Create a bar chart to compare crime volume across the four time periods.
px.bar(
    time_of_day_counts,
    x="time_of_day",
    y="incident_count",
    title="Recorded Crime Incidents by Broad Time of Day",
    labels={"time_of_day": "Time of Day", "incident_count": "Number of Incidents"},
    text_auto=True
)

## 7. Victim Demographics

The cleaning notebook converted clearly invalid victim ages to missing and validated the remaining values. This section examines the observed victim-age distribution and the coded victim sex/descent distributions.

Demographic fields contain source missingness, so percentages are interpreted carefully and missing values are shown when they are analytically useful.


In [ ]:
# Select valid victim ages, excluding missing values.
age_distribution = df.loc[df["Vict Age"].notna(), ["Vict Age"]]

# Find the most common recorded victim age.
peak_age = age_distribution["Vict Age"].mode()[0]

# Create a histogram to visualize the distribution of recorded victim ages.
px.histogram(
    age_distribution,
    x="Vict Age",
    nbins=100,
    title="Distribution of Recorded Victim Ages",
    labels={
        "Vict Age": "Victim Age",
        "count": "Number of Incidents"
    },
).add_vline(
    x=peak_age,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Most common age: {peak_age}",
    annotation_position="top"
)


In [ ]:
# Count incidents by victim sex, treating missing values as a separate category.
sex_counts = (
    df["Vict Sex"].fillna("Missing")
    .value_counts()
    .rename_axis("Vict Sex")
    .reset_index(name="incident_count")
)

# Display the incident counts for each victim sex category.
display(sex_counts)

# Create a bar chart to compare incident counts across victim sex categories.
px.bar(
    sex_counts,
    x="Vict Sex",
    y="incident_count",
    title="Recorded Incidents by Victim Sex Code",
    labels={
        "Vict Sex": "Victim Sex Code",
        "incident_count": "Number of Incidents"
    },
    text_auto=True
)

In [ ]:
# Define a mapping from LAPD victim-descent codes to descriptive labels.
descent_mapping = {
    "A": "Other Asian",
    "B": "Black",
    "C": "Chinese",
    "D": "Cambodian",
    "F": "Filipino",
    "G": "Guamanian",
    "H": "Hispanic/Latin/Mexican",
    "I": "American Indian/Alaskan Native",
    "J": "Japanese",
    "K": "Korean",
    "L": "Laotian",
    "O": "Other",
    "P": "Pacific Islander",
    "S": "Samoan",
    "U": "Hawaiian",
    "V": "Vietnamese",
    "W": "White",
    "X": "Unknown",
    "Z": "Asian Indian",
}

# Create a descriptive column for analysis while preserving the original codes.
# Missing values remain missing if they do not match a mapping key.
df["Vict Descent Desc"] = df["Vict Descent"].map(descent_mapping)

# Create and display a lookup table to verify the code-to-label mapping.
descent_lookup = pd.DataFrame(
    list(descent_mapping.items()), columns=["Vict Descent", "Vict Descent Desc"])

display(descent_lookup)

In [ ]:
# Count incidents by victim descent using descriptive labels.
# Treat missing values separately from the explicit "Unknown" category.

descent_counts = (
    df["Vict Descent Desc"]
    .fillna("Missing")
    .value_counts()
    .rename_axis("Vict Descent")
    .reset_index(name="incident_count")
)

# Display the incident counts for each victim descent category.
display(descent_counts)

# Create a bar chart to compare incident counts across victim descent categories.
px.bar(
    descent_counts,
    x="Vict Descent",
    y="incident_count",
    text_auto=True,
    title="Recorded Incidents by Victim Descent",
    labels={
        "Vict Descent": "Victim Descent",
        "incident_count": "Number of Incidents",
    },
)

In [ ]:
# Compare victim-age distributions across the five most common crime types.
# Exclude missing ages and sample up to 25,000 incidents per crime type.

age_by_crime = df[
    df["Crm Cd Desc"].isin(top5_crimes) & df["Vict Age"].notna()
][["Crm Cd Desc", "Vict Age"]]

sample_size = min(25000, age_by_crime.groupby("Crm Cd Desc").size().min())

age_sample = (
    age_by_crime.groupby("Crm Cd Desc", group_keys=False)
    .sample(n=sample_size, random_state=42)
)

# Visualize and compare victim-age distributions across crime types.
px.box(
    age_sample,
    x="Crm Cd Desc",
    y="Vict Age",
    points=False,
    title="Victim Age Distribution for the Five Most Common Crime Types",
    labels={"Crm Cd Desc": "Crime Type", "Vict Age": "Victim Age"},
).update_yaxes(range=[1, 100]).update_xaxes(tickangle=-30).show()

## 8. Premise and Weapon Patterns

`Premis Desc` and `Weapon Desc` contain source missingness. Missing values are therefore preserved as a meaningful EDA category rather than automatically treated as zero or "none".

The charts below focus on the most frequently recorded categories.


In [ ]:
# Identify the most common premise types, including missing descriptions.
top_premises = (
    df["Premis Desc"].fillna("Missing")
    .value_counts()
    .rename_axis("Premis Desc")
    .reset_index(name="incident_count")
    .head(TOP_N)
)

# Display the top premise types and their incident counts.
display(top_premises)

# Create a horizontal bar chart to compare incident counts by premise type.
px.bar(
    top_premises.sort_values("incident_count"),
    x="incident_count",
    y="Premis Desc",
    orientation="h",
    text_auto=True,
    title=f"Top {TOP_N} Recorded Premise Types",
    labels={
        "Premis Desc": "Premise",
        "incident_count": "Number of Incidents"
    },
)

In [ ]:
# Identify the most frequently recorded weapon descriptions, including missing values.
top_weapons = (
    df["Weapon Desc"].fillna("Missing")
    .value_counts()
    .rename_axis("Weapon Desc")
    .reset_index(name="incident_count")
    .head(TOP_N)
)

# Display the top weapon descriptions and their incident counts.
display(top_weapons)

# Create a horizontal bar chart to compare recorded weapon descriptions by incident count.
px.bar(
    top_weapons.sort_values("incident_count"),
    x="incident_count",
    y="Weapon Desc",
    orientation="h",
    text_auto=True,
    title=f"Top {TOP_N} Recorded Weapon Descriptions",
    labels={
        "Weapon Desc": "Weapon Description",
        "incident_count": "Number of Incidents"
    },
)

In [ ]:
# Compare the five most common crime types with the five most common premise types.
top5_premises = top_premises.loc[
    top_premises["Premis Desc"] != "Missing", "Premis Desc"
].head(5).tolist()

# Count incidents for each combination of crime type and premise type.
crime_premise = (
    df[df["Crm Cd Desc"].isin(top5_crimes) & df["Premis Desc"].isin(top5_premises)]
    .groupby(["Crm Cd Desc", "Premis Desc"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
)

# Create a heatmap to compare incident counts across crime and premise types.
px.density_heatmap(
    crime_premise,
    x="Premis Desc",
    y="Crm Cd Desc",
    z="incident_count",
    histfunc="sum",
    text_auto=True,
    title="Incident Counts for Common Crime Types and Premise Types",
    labels={
        "Premis Desc": "Premise",
        "Crm Cd Desc": "Crime Type",
        "incident_count": "Incidents"
    },
).update_xaxes(tickangle=-30)

## 9. Reporting Lag

`reporting_lag_days` measures the difference between `Date Rptd` and `DATE OCC` in days. The cleaning notebook removed negative reporting lags because a report date preceding the occurrence date is inconsistent with the intended interpretation of the fields.

EDA here focuses on the distribution and the relationship between reporting lag and occurrence year.


In [ ]:
# Summarize reporting delays using descriptive statistics, excluding missing values.
reporting_lag_summary = df["reporting_lag_days"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

# Display the summary statistics as a DataFrame.
display(reporting_lag_summary.to_frame("reporting_lag_days"))

## 10. Geographic Patterns

Latitude and longitude are available for many incidents, but the cleaning notebook preserved missing coordinates rather than removing the incident rows.

This section first examines coordinate coverage and then uses aggregated area-level patterns. A full three-million-point geographic scatterplot would be unnecessarily heavy, so the visualization uses a filtered sample for point-level context.


In [ ]:
# Calculate the completeness of geographic coordinates in the dataset.
coordinate_complete = df["LAT"].notna() & df["LON"].notna()

# Summarize total records, coordinate availability, and completeness percentage.
coordinate_summary = pd.DataFrame({
    "records": [len(df)],
    "complete_coordinates": [coordinate_complete.sum()],
    "missing_coordinates": [(~coordinate_complete).sum()],
    "complete_percent": [coordinate_complete.mean() * 100],
})

# Display the geographic coverage summary.
display(coordinate_summary)

In [ ]:
# Sample up to 10,000 records with complete coordinates for geographic visualization.
geo_sample = (
    df.loc[coordinate_complete, ["LAT", "LON", "Crm Cd Desc", "year", "AREA NAME"]]
      .sample(n=min(10000, coordinate_complete.sum()), random_state=42)
)

# Plot crime locations on an interactive map, colored by LAPD area.
px.scatter_map(
    geo_sample,
    lat="LAT",
    lon="LON",
    color="AREA NAME",
    hover_data=["Crm Cd Desc", "year"],
    zoom=9,
    height=650,
    title="Sample of Recorded Crime Locations by LAPD Area",
)

In [ ]:
# Compare the five most common crime types across the ten LAPD areas with the highest incident counts.
top10_areas = area_counts.head(10)["AREA NAME"].tolist()

# Count incidents for each crime type within each selected area.
area_crime = (
    df[df["AREA NAME"].isin(top10_areas) & df["Crm Cd Desc"].isin(top5_crimes)]
    .groupby(["AREA NAME", "Crm Cd Desc"], as_index=False)
    .size()
    .rename(columns={"size": "incident_count"})
)

# Create a grouped bar chart to compare crime counts across LAPD areas.
px.bar(
    area_crime,
    x="AREA NAME",
    y="incident_count",
    color="Crm Cd Desc",
    barmode="group",
    title="Common Crime Types Across the Ten Highest-Volume LAPD Areas",
    labels={
        "AREA NAME": "LAPD Area",
        "incident_count": "Number of Incidents",
        "Crm Cd Desc": "Crime Type"
    },
).update_xaxes(tickangle=-30)

## 11. Cross-Dimension Exploration

The following views combine two or more cleaned variables to identify patterns that may not be visible in one-dimensional summaries.

These charts are intentionally descriptive. A visible relationship should be treated as a candidate for further investigation rather than evidence of causation.

In [ ]:
# Compare recorded crime counts by hour and day of the week.
hour_weekday = (
    df.groupby(["day_of_week", "day_name", "hour"], as_index=False)
      .size()
      .rename(columns={"size": "incident_count"})
      .sort_values(["day_of_week", "hour"])
)

# Create a heatmap to identify patterns in crime volume across days and hours.
px.density_heatmap(
    hour_weekday,
    x="hour",
    y="day_name",
    z="incident_count",
    histfunc="sum",
    text_auto=True,
    title="Recorded Crime Volume by Day of Week and Hour",
    labels={
        "hour": "Hour of Day",
        "day_name": "Day of Week",
        "incident_count": "Incidents"
    },
).update_xaxes(dtick=1)

## 12. EDA Summary Tables

These compact tables make the main distributions available without requiring the reader to infer exact values from charts. They are also useful checkpoints before writing the narrative findings for the project report.

In [ ]:

# Build a compact set of headline statistics for later interpretation.
# No data is modified in this section; the goal is to create reusable numerical summaries.

eda_headlines = {
    "total_incidents": len(df),
    "study_start": df["DATE OCC"].min(),
    "study_end": df["DATE OCC"].max(),
    "unique_crime_types": df["Crm Cd Desc"].nunique(dropna=True),
    "unique_lapd_areas": df["AREA NAME"].nunique(dropna=True),
    "median_victim_age": df["Vict Age"].median(),
    "median_reporting_lag_days": df["reporting_lag_days"].median(),
    "coordinate_coverage_percent": coordinate_complete.mean() * 100,
}

display(pd.Series(eda_headlines, name="value").to_frame())


In [ ]:
# Display the top 10 crime types, LAPD areas, and premise categories for the EDA summary.

print("Top 10 crime types")
display(
    df["Crm Cd Desc"].value_counts().head(10)
    .rename_axis("Crm Cd Desc")
    .reset_index(name="incident_count")
)

In [ ]:
print("Top 10 LAPD areas")
display(area_counts.head(10))

In [ ]:
print("Top 10 premise categories")
display(
    df["Premis Desc"].fillna("Missing").value_counts().head(10)
    .rename_axis("Premis Desc")
    .reset_index(name="incident_count")
)